# Learning-Rate Pilotu — Kalan 60 Koşu

Tam 120 LR taramasının ikinci yarısıdır. **Leaky ReLU + ELU + Swish × 5 optimizer × 4 LR = 60 koşu**. Her koşu 30 epoch, seed 42 ve CIFAR-10 validation protokolüyle çalışır.

İlk 60 sonuçlarına dokunulmaz. Bu notebook sonuçları ayrı Drive klasörüne yazar. İkinci 60 bittiğinde ilk ve ikinci 60 birlikte analiz edilerek 5 optimizer için nihai LR değerleri seçilir. Test seti kullanılmaz.


In [ ]:
import subprocess
subprocess.run(['nvidia-smi'], check=True)


## Güncel GitHub sürümünü al


In [ ]:
import io, os, shutil, tarfile, requests, json
from pathlib import Path

OWNER='erdemci07'; REPO='derin-ogrenme-egitim-dinamikleri'; BRANCH='main'
repo_dir=Path('/content/derin-ogrenme-egitim-dinamikleri')
extract_root=Path('/content/_tez_repo_extract')
shutil.rmtree(repo_dir, ignore_errors=True)
shutil.rmtree(extract_root, ignore_errors=True)
extract_root.mkdir(parents=True, exist_ok=True)
r=requests.get(f'https://api.github.com/repos/{OWNER}/{REPO}/commits/{BRANCH}', timeout=60)
r.raise_for_status(); commit_sha=r.json()['sha']
a=requests.get(f'https://api.github.com/repos/{OWNER}/{REPO}/tarball/{commit_sha}', timeout=120)
a.raise_for_status()
with tarfile.open(fileobj=io.BytesIO(a.content), mode='r:gz') as tar:
    tar.extractall(extract_root)
kokler=[p for p in extract_root.iterdir() if p.is_dir()]
if len(kokler)!=1:
    raise RuntimeError('GitHub arşiv yapısı beklenenden farklı.')
shutil.move(str(kokler[0]), str(repo_dir)); os.chdir(repo_dir)
os.environ['TEZ_GIT_COMMIT']=commit_sha
print('Repo hazır:',commit_sha)


## Google Drive yolları


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_KOK=Path('/content/drive/MyDrive')
DRIVE_VERI=DRIVE_KOK/'tez_datasets'
DRIVE_PROJE=DRIVE_KOK/'tez_deney_sonuclari'
DRIVE_AYAR=DRIVE_PROJE/'ortak_ayarlar'
DRIVE_ILK60=DRIVE_PROJE/'lr_pilotu'
DRIVE_IKINCI60=DRIVE_PROJE/'lr_pilotu_kalan_60'
DRIVE_ANALIZ120=DRIVE_PROJE/'lr_pilotu_120_analiz'
if not DRIVE_VERI.is_dir():
    raise FileNotFoundError(f'Veri klasörü bulunamadı: {DRIVE_VERI}')
if not DRIVE_ILK60.is_dir():
    raise FileNotFoundError(f'İlk 60 sonuç klasörü bulunamadı: {DRIVE_ILK60}')
DRIVE_AYAR.mkdir(parents=True, exist_ok=True)
DRIVE_IKINCI60.mkdir(parents=True, exist_ok=True)
DRIVE_ANALIZ120.mkdir(parents=True, exist_ok=True)
SPLIT_DOSYASI=DRIVE_AYAR/'cifar10_split.json'
STATS_DOSYASI=DRIVE_AYAR/'cifar10_stats.json'
os.environ['TEZ_VERI_KLASORU']=str(DRIVE_VERI)
os.environ['TEZ_SPLIT_DOSYASI']=str(SPLIT_DOSYASI)
os.environ['TEZ_STATS_DOSYASI']=str(STATS_DOSYASI)
os.environ['TEZ_CIKTI_KLASORU']=str(DRIVE_IKINCI60)
print('İlk 60:',DRIVE_ILK60)
print('İkinci 60 çıktıları:',DRIVE_IKINCI60)


## CUDA ve bağımlılıklar


In [ ]:
subprocess.run(['python','-m','pip','install','-q','-r','requirements_colab.txt'], check=True)
import torch
print('PyTorch:',torch.__version__)
print('CUDA:',torch.cuda.is_available())
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'YOK')
if not torch.cuda.is_available():
    raise RuntimeError('T4/CUDA aktif değil.')


## Kalan 60 koşuyu çalıştır / devam ettir

NaN/Inf üreten bir aday **kararsız/uygunsuz** olarak kaydedilir ve sonraki çalıştırmada yeniden denenmez. Tamamlanan koşular atlanır; gerçek checkpoint varsa kaldığı yerden devam eder.


In [ ]:
import importlib
import calistir_lr_pilotu as lr
importlib.reload(lr)
lr.ana(['--calistir','--ayarlar-dosyasi','ayarlar/lr_pilotu_kalan_60.yaml'])


## İkinci 60 ilerleme özeti


In [ ]:
def benzersiz_durum(kok):
    tamam={}; kararsiz={}
    for p in kok.glob('*/tamamlandi.json'):
        try:
            d=json.loads(p.read_text(encoding='utf-8')); tamam[d['run_id']]=d
        except Exception:
            pass
    for p in kok.glob('*/kararsiz.json'):
        try:
            d=json.loads(p.read_text(encoding='utf-8'))
            if d['run_id'] not in tamam:
                kararsiz[d['run_id']]=d
        except Exception:
            pass
    return tamam, kararsiz

tamam2, kararsiz2 = benzersiz_durum(DRIVE_IKINCI60)
deger2=len(tamam2)+len(kararsiz2)
print(f'İkinci 60: {deger2}/60 değerlendirildi ({len(tamam2)} sağlıklı, {len(kararsiz2)} kararsız/uygunsuz)')
if deger2 < 60:
    print('Kesinti olursa aynı notebooku yeniden Run all yapın.')


## 120/120 tamamlandıysa nihai LR seçimini yap

Seçim altı aktivasyonun tamamı üzerinden yapılır: birincil ölçüt 30. epoch ortalama validation accuracy, ikincil ölçüt ortalama validation loss; tam eşitlikte daha düşük LR seçilir.


In [ ]:
if deger2 == 60:
    import analiz_lr_pilotu_120 as analiz120
    importlib.reload(analiz120)
    analiz120.ana([
        '--ilk-60-klasoru',str(DRIVE_ILK60),
        '--ikinci-60-klasoru',str(DRIVE_IKINCI60),
        '--cikti-klasoru',str(DRIVE_ANALIZ120),
    ])
else:
    print('120 koşuluk nihai LR analizi için ikinci 60 henüz bitmedi.')
